# Cross Platform Pbmc

One original PBMC pair (10x Chromium V2 A to V3); this does not run all 72 pairs. Source: `MPNN_SelfDistill/benchmark/cross_platform(small)/batch_batch/platform.ipynb`. Only the original training and metric steps are retained; figure generation is omitted. Package imports, extracted-data paths, run output paths, and the CUDA device are the only training-cell adaptations.

Original training configuration: GNN and batch follow the original adaptive rule; max_steps=500, high=True, rate=0.05.

Extract the matching restricted Zenodo archive into `data/experiments/` before running. Use `SCPATH_STUDY_DATA` to point to a different extraction directory and `SCPATH_DEVICE` to select a GPU. Set `PYTHONHASHSEED=42` before starting Jupyter, because the source preprocessing uses set-based gene intersections. Run cells in order with the `scPath` kernel. Historical result files in the archive are for comparison only; this notebook writes new outputs under `output/experiments/`.


In [ ]:
import os
import sys
from pathlib import Path
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scpath/models.py").is_file())
sys.path.insert(0, str(repo_root))
data_root = Path(os.environ.get("SCPATH_STUDY_DATA", str(repo_root / "data/experiments")))
study_data = data_root / "cross_platform_pbmc/original_paths/data/pairs/10x_Chromium_V2_A_to_10x_Chromium_V3"
run_dir = repo_root / "output/experiments/cross_platform_pbmc"
if not study_data.is_dir():
    raise FileNotFoundError(f"Extract the cross_platform_pbmc Zenodo archive under {data_root}")
run_dir.mkdir(parents=True, exist_ok=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this historical training workflow")
print("GPU:", torch.cuda.get_device_name(torch.device(os.environ.get("SCPATH_DEVICE", "cuda:0"))))


In [ ]:
import sys
sys.path.insert(0, str(repo_root))
from scpath.utils.data_preprocess import data_processing
from scpath.utils.pathway import built_pathway_graphs
from scpath.utils.data_preprocess import preprocess
from scpath.train import train
import torch
import os
from scpath.config import Config
import pandas as pd
import scipy.sparse as sparse
from scipy.io import mmread, mmwrite
import random
import numpy as np
import torch 

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42) 
exp_path = str(run_dir) + "/"

train_path = str(study_data / "train") + "/"
test_path = str(study_data / "test") + "/"
path_output = f'{exp_path}/output/'
species_train='human'  
species_test='human'
plot=False
save_data=False
os.makedirs(path_output, exist_ok=True)


adata_train_rna, adata_test_rna = data_processing(train_path, test_path, path_output, species_train, species_test, plot, save_data)


def get_adaptive_strategy(n_train, n_test, estimated_epochs=50):
    total_cells = n_train + n_test
    min_cells = min(n_train, n_test)
    max_cells = max(n_train, n_test)
    
    
    if total_cells < 10000:
        gnn_type = 'gat'
    elif 10000 <= total_cells <= 50000:
        gnn_type = 'gcn'
    else:
        gnn_type = 'sage'

    
    if min_cells < 1000:
        batch_size = min_cells//2
    elif min_cells < 3000:
        batch_size = 512
    else:
        batch_size = 1024
        
    
    
    if batch_size > min_cells:
        batch_size = min_cells

    
    
    steps_per_epoch = max(1, min(max_cells //batch_size,(min_cells//batch_size)*2))
    target_steps = int(estimated_epochs * steps_per_epoch)
    
    if (target_steps//(n_train//batch_size))<estimated_epochs:
        target_steps=estimated_epochs*(n_train//batch_size)
    return gnn_type, batch_size, target_steps



n_train = adata_train_rna.n_obs
n_test = adata_test_rna.n_obs

gnn_type, batch_size, effective_steps = get_adaptive_strategy(n_train, n_test)
effective_steps=500
print("="*60)
print(f"Adaptive strategy (total cells: {n_train + n_test})")
print(f" - Train: {n_train}, Test: {n_test}")
print(f" - GNN Type:   {gnn_type}")
print(f" - Batch Size: {batch_size}")
print(f" - Max Steps:  {effective_steps}")
print("="*60)


pathway_names=['Reactome','KEGG']
file_path = str(repo_root / "data/pathway/human") + "/" 
built_pathway_graphs(file_path, pathway_names, adata_train_rna, adata_test_rna, path_output, n_neighbors=5)


adata_train_rna, hvg_genes = preprocess(adata_train_rna)
adata_test_rna = preprocess(adata_test_rna, hvg_genes=hvg_genes)


device = torch.device(os.environ.get("SCPATH_DEVICE", "cuda:0"))
config = Config(
    f'{exp_path}/',
    adata_train_rna.X.shape[1],
    device,
    True,
    batch_size=int(batch_size),
    gnn_type=gnn_type,
    max_steps=effective_steps,
    K=len(adata_test_rna.obs['cell_type'].unique())
    ,rate=0.05
)

print(f"Config Batch Size Type: {type(config.batch_size)}")
train(config, adata_train_rna, adata_test_rna, pathway_names)

In [ ]:
# Metrics follow the original prediction columns; no retraining is done here.
import scanpy as sc
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report
adata_test_result = sc.read_h5ad(run_dir / "logs/test_results_distilled.h5ad")
y_true = adata_test_result.obs["labels"]
y_pred = adata_test_result.obs["final_distill_preds"]
accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"Overall accuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(pd.DataFrame(classification_report(y_true, y_pred, output_dict=True)))
